# Mohs Hardness Prediction — Baseline
**AOAI Task 3** · Regression · Tabular

Each row describes a mineral through averaged physical and chemical properties of its elements
(electron counts, density, ionisation energy, electronegativity, atomic radii, ...).
Your job is to predict its **Mohs hardness** — the scratch-resistance scale that goes from
1 (talc) to 10 (diamond).

| | |
|---|---|
| **Target** | `Hardness` (continuous, 1–10) |
| **Metric** | **Median Absolute Error** (lower is better) |
| **Submission** | `id,Hardness` — one row per test id |

### Notebook structure
1. Setup
2. Load data
3. Quick EDA
4. Feature engineering
5. LightGBM with K-fold cross-validation
6. Generate submission
7. Ideas to improve

## 1. Setup

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, median_absolute_error

warnings.filterwarnings('ignore')

SEED   = 42
TARGET = 'Hardness'
np.random.seed(SEED)

# Works on Kaggle and locally (put the competition files in ./data)
COMPETITION = 'moth-hardness-task'
CANDIDATES  = [Path(f'/kaggle/input/competitions/{COMPETITION}'), Path(f'/kaggle/input/{COMPETITION}'), Path('data')]
DATA_DIR    = next(p for p in CANDIDATES if p.exists())
print('Data directory:', DATA_DIR)

## 2. Load Data

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Train: {train.shape} | Test: {test.shape}')
train.head()

## 3. Quick EDA

In [ ]:
print(train[TARGET].describe())
print('\nMissing values in train:', int(train.isnull().sum().sum()))

fig, ax = plt.subplots(figsize=(10, 4))
train[TARGET].hist(bins=60, ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Distribution of Mohs hardness')
ax.set_xlabel('Hardness')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

A few interaction and ratio features between the physical properties.

In [ ]:
DROP = ['id']

def engineer(df):
    df = df.copy()
    # Interaction features
    df['density_x_ionenergy'] = df['density_Total']         * df['ionenergy_Average']
    df['elneg_x_val_e']       = df['el_neg_chi_Average']    * df['val_e_Average']
    df['atomic_x_density']    = df['atomicweight_Average']  * df['density_Average']
    df['rcov_x_rvdw']         = df['R_cov_element_Average'] * df['R_vdw_element_Average']
    df['zaratio_x_ionenergy'] = df['zaratio_Average']       * df['ionenergy_Average']
    # Ratio features
    df['density_ratio']       = df['density_Total']      / (df['density_Average'] + 1e-9)
    df['electron_ratio']      = df['allelectrons_Total'] / (df['allelectrons_Average'] + 1e-9)
    return df

train = engineer(train)
test  = engineer(test)

feature_cols = [c for c in train.columns if c not in DROP + [TARGET]]
X      = train[feature_cols]
y      = train[TARGET]
X_test = test[feature_cols]

print(f'Features: {len(feature_cols)} | Train rows: {len(X)}')

## 5. LightGBM — K-Fold Cross-Validation

We train one model per fold, score each fold on its held-out part (out-of-fold, "OOF"),
and average the fold models' predictions on the test set.

The competition is scored with **Median** Absolute Error, so that is the number to watch.

In [ ]:
params = dict(
    n_estimators      = 3000,
    learning_rate     = 0.03,
    num_leaves        = 31,
    min_child_samples = 20,
    subsample         = 0.8,
    subsample_freq    = 1,
    colsample_bytree  = 0.8,
    random_state      = SEED,
    n_jobs            = -1,
    verbose           = -1,
)

N_FOLDS    = 5
kf         = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_preds  = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y), start=1):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )
    oof_preds[val_idx] = model.predict(X_val)
    test_preds        += model.predict(X_test) / N_FOLDS

    print(f'Fold {fold} | MedAE: {median_absolute_error(y_val, oof_preds[val_idx]):.4f} '
          f'| Best iter: {model.best_iteration_}')

oof_medae = median_absolute_error(y, oof_preds)
print(f'\nOOF Median AE: {oof_medae:.4f}   (MAE: {mean_absolute_error(y, oof_preds):.4f})')

## 6. Generate Submission

In [ ]:
# Clip to the valid Mohs scale
test_preds = np.clip(test_preds, 1.0, 10.0)

submission = pd.DataFrame({'id': test['id'], TARGET: test_preds})
submission.to_csv('submission.csv', index=False)

print(f'Saved {len(submission)} predictions to submission.csv')
submission.head()

## 7. Ideas to Improve

- **Match the loss to the metric.** The model above minimises squared error, but you are scored
on the *median* absolute error. Which LightGBM `objective` targets that?
- **Look closely at the target.** Plot the most frequent hardness values — is the target really continuous?
- More / better features, hyperparameter tuning, other models (XGBoost, CatBoost) and blending.